In [2]:
import asyncio
from functools import partial
from pathlib import Path

import ipywidgets as widgets
import pygame
from IPython.display import display
from ipyevents import Event

# Clean up background activity when this cell is rerun.
if "intro_monitor_task" in globals():
    intro_monitor_task.cancel()

if "key_events" in globals():
    for listener in key_events:
        listener.close()

# Local audio folders
audio_dir = Path(r"D:\KAOS\visionNov132026-EDITS\audio")
tones_dir = audio_dir / "phoneTones"
recordings_dir = audio_dir / "weLoveLeimertArchive-Nov132026"

pygame.mixer.init()
pygame.mixer.stop()

recordings = {
    "0": "noPriorArt1.wav",
    "1": "AmaraKiona.wav",
    "2": "BenCaldwell.wav",
    "3": "Benu.wav",
    "4": "HughAugustine.wav",
    "5": "Jimetta.wav",
    "6": "MaryChristian.wav",
    "7": "MattGibson.wav",
    "8": "Mitebrafoagindotan.wav",
    "9": "WeLoveLeimertAudioCollage.wav",
}

# Load short tones in advance so presses respond quickly.
key_sounds = {
    key: pygame.mixer.Sound(str(tones_dir / f"DTMF-{key}.wav"))
    for key in recordings
}

intro_sound = pygame.mixer.Sound(str(recordings_dir / recordings["0"]))
channel = pygame.mixer.Channel(0)

phone_off_hook = False
active_key = None

phone_output = widgets.Output()


def pick_up():
    global phone_off_hook

    if phone_off_hook:
        return

    pygame.mixer.stop()
    phone_off_hook = True
    channel.play(intro_sound, loops=-1)
    print(f"Phone picked up. Playing intro: {recordings['0']}")


def hang_up():
    global phone_off_hook, active_key

    phone_off_hook = False
    active_key = None
    pygame.mixer.stop()
    print("Phone hung up. Audio stopped.")


def press_key(key):
    global active_key

    if not phone_off_hook:
        print("Pick up the phone before selecting a recording.")
        return

    if active_key is not None:
        return

    pygame.mixer.stop()
    active_key = key
    channel.play(key_sounds[key], loops=-1)
    print(f"Pressed: {key}")


def release_key(key):
    global active_key

    if active_key != key:
        return

    # Stop the tone immediately on release.
    channel.stop()
    active_key = None

    if not phone_off_hook:
        return

    filename = recordings[key]
    recording = pygame.mixer.Sound(str(recordings_dir / filename))
    channel.play(recording)
    print(f"Released: {key}. Playing: {filename}")


def cancel_key(key):
    global active_key

    if active_key == key:
        channel.stop()
        active_key = None
        print(f"Cancelled key: {key}")


async def monitor_playback():
    while True:
        await asyncio.sleep(0.1)

        if (
            phone_off_hook
            and active_key is None
            and not channel.get_busy()
        ):
            channel.play(intro_sound, loops=-1)
            with phone_output:
                print("Playback idle. Returning to intro.")


# Pickup/hang-up toggle
hook_toggle = widgets.ToggleButton(
    value=False,
    description="Pick Up",
    layout=widgets.Layout(width="222px"),
)


def on_hook_change(change):
    with phone_output:
        if change["new"]:
            pick_up()
            hook_toggle.description = "Hang Up"
        else:
            hang_up()
            hook_toggle.description = "Pick Up"


hook_toggle.observe(on_hook_change, names="value")


# Mouse press/release handling
def on_key_event(event, key):
    event_type = event["type"]

    # Ignore non-left mouse buttons.
    if event_type in ("mousedown", "mouseup"):
        if event.get("button", 0) != 0:
            return

    with phone_output:
        if event_type == "mousedown":
            press_key(key)
        elif event_type == "mouseup":
            release_key(key)
        elif event_type == "mouseleave":
            cancel_key(key)


# Build the keypad.
rows = []
key_events = []

for keys in [
    ["1", "2", "3"],
    ["4", "5", "6"],
    ["7", "8", "9"],
    ["*", "0", "#"],
]:
    buttons = []

    for key in keys:
        button = widgets.Button(
            description=key,
            layout=widgets.Layout(width="70px", height="50px"),
            disabled=key in ("*", "#"),
        )

        if key not in ("*", "#"):
            listener = Event(
                source=button,
                watched_events=["mousedown", "mouseup", "mouseleave"],
                prevent_default_action=True,
            )
            listener.on_dom_event(partial(on_key_event, key=key))
            key_events.append(listener)

        buttons.append(button)

    rows.append(widgets.HBox(buttons))

keypad = widgets.VBox(rows)
phone_controls = widgets.VBox([
    hook_toggle,
    keypad,
    phone_output,
])

intro_monitor_task = asyncio.create_task(monitor_playback())

display(phone_controls)

with phone_output:
    print("Prototype ready. Phone is hung up.")